In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd

2026-03-01 21:45:41.841909: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-01 21:45:41.876503: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-01 21:45:41.877071: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-01 21:45:42.592811: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


### **Carga y limpieza de datos:**

Dentro del conjunto de datos seleccionado, las variables *transaction_id* y *user_id* se consideran irrelevantes para el entrenamiento del modelo. En el caso de transaction_id, su inclusión podría inducir overfitting, ya que se trata de un identificador único sin valor predictivo real. Por su parte, user_id requeriría un volumen significativamente mayor de información histórica para modelar patrones de reincidencia de manera confiable, por lo que también se excluye del análisis.

Las variables *(transaction_amount, account_age_days, transaction_hour, previous_failed_attempts, avg_transaction_amount, ip_risk_score, login_attempts_last_24h)* corresponden a datos numéricos continuos o discretos, los cuales son directamente utilizables y evaluables por la red neuronal tras un proceso adecuado de normalización o estandarización.

En contraste, las variables *(transaction_type, payment_mode, device_type, device_location)* son de tipo categórico (string). Para poder incorporarlas al modelo, se transforman mediante un mapeo numérico apropiado, permitiendo su procesamiento dentro de la arquitectura neuronal.

La variable *is_international* es de naturaleza binaria y puede ser utilizada directamente como característica de entrada, ya que representa información relevante en formato compatible con el modelo.

En consecuencia, todas las variables mencionadas constituyen los features del modelo. La variable objetivo se encuentra en la última columna bajo el nombre *fraud_label*, la cual representa una salida binaria que indica si la transacción corresponde o no a un caso de fraude.

In [2]:
df=pd.read_csv('Datasets/Digital_Payment_Fraud_Detection_Dataset.csv')
unique_ids=df['transaction_id'].unique()
# Reduccion de valores negativos para compensar la desigualdad
df=df[(df['transaction_id'].isin(unique_ids[0:int(len(unique_ids)*0.2)]))|(df['fraud_label']==1)]
#cols_str= [transaction_type, payment_mode, device_type, device_location]
#print(df.dtypes)
df=df.drop(columns=['transaction_id', 'user_id'])

for i in df.columns:
    if df[i].dtype== object:
        unicos=df[i].unique()
        mapeo={}
        c=0
        for j in unicos:
            mapeo[j]=c
            c+=1
        #print(i, df[i].dtype, mapeo)
        df[f'{i}_cat']=df[i].replace(mapeo)
        df=df.drop(columns=i)

df.head(2)

,transaction_amount,account_age_days,transaction_hour,previous_failed_attempts,avg_transaction_amount,is_international,ip_risk_score,login_attempts_last_24h,fraud_label,transaction_type_cat,payment_mode_cat,device_type_cat,device_location_cat
0,18758.28,895,14,1,25535.84,0,0.718,4,0,0,0,0,0
1,47538.18,918,21,0,3955.85,0,0.525,9,0,1,1,1,0


### **Datos de ensayo y verificación**

In [3]:
rand_sel=np.random.rand(len(df))*7
rand_sel=rand_sel.astype(int)
df['random']=rand_sel
df_princ=df[df['random']<4].copy().drop(columns='random')
df_1=df[df['random']==4].copy().drop(columns='random')
df_2=df[df['random']==5].copy().drop(columns='random')
df_3=df[df['random']==6].copy().drop(columns='random')

In [4]:
print(  len(df_princ[df_princ['fraud_label']==1]),
        len(df_princ[df_princ['fraud_label']==0]), '\n',
        len(df_1[df_1['fraud_label']==1]), 
        len(df_1[df_1['fraud_label']==0]), '\n',
        len(df_2[df_2['fraud_label']==1]), 
        len(df_2[df_2['fraud_label']==0]), '\n',
        len(df_3[df_3['fraud_label']==1]),
        len(df_3[df_3['fraud_label']==0]))

287 796 
 69 216 
 67 211 
 66 184


### **Definición del modelo:**

Se procede a definir el modelo, con los datos categorizados y la unica salida binaria

In [5]:
x=[]
for i in df_princ.columns:
    if i!= 'fraud_label':
        x.append(df_princ[i])

X=np.column_stack(x)
tamaño= len(x)

In [10]:
entrada = tf.keras.layers.Dense(units=tamaño, input_shape=[tamaño])
c1 = tf.keras.layers.Dense(units=tamaño)
c2 = tf.keras.layers.Dense(units=tamaño)
c3 = tf.keras.layers.Dense(units=tamaño)
salida = tf.keras.layers.Dense(units=1)
red = tf.keras.Sequential([entrada, c1, c2, c3, salida])
red.compile(
    optimizer=tf.keras.optimizers.Adam(0.1),
    loss='binary_crossentropy',
metrics=[
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.AUC(name='auc_roc'),
]

)

In [13]:
historial = red.fit(X, df_princ['fraud_label'], epochs=1000, verbose=False)

In [14]:
fila=25
cols_ver=['transaction_amount', 'account_age_days', 'transaction_hour', 'previous_failed_attempts', 'avg_transaction_amount', 'is_international', 'ip_risk_score', 'login_attempts_last_24h', 'transaction_type_cat', 'payment_mode_cat','device_type_cat','device_location_cat']
k=df_1[cols_ver].iloc[[fila]].values

k2=df_1[['fraud_label']].iloc[fila].values
print(k,'\n'*2, k2)
red.predict(k)

[[7.287500e+03 1.535000e+03 8.000000e+00 3.000000e+00 2.567155e+04
  0.000000e+00 1.360000e-01 3.000000e+00 1.000000e+00 3.000000e+00
  1.000000e+00 4.000000e+00]] 

 [0]
1/1 [==============================] - 0s 15ms/step


array([[7506.365]], dtype=float32)

In [15]:

def clasificar_dataframe(df, modelo, columnas):
    X = df[columnas].values
    preds = modelo.predict(X)
    df['clasificacion'] = preds#(preds > 0.5).astype(int)
    return df

df_1= clasificar_dataframe(df_1, red, cols_ver)

df_1[df_1['clasificacion']<-0.5].reset_index().head(5)

9/9 [==============================] - 0s 871us/step


,index,transaction_amount,account_age_days,transaction_hour,previous_failed_attempts,avg_transaction_amount,is_international,ip_risk_score,login_attempts_last_24h,fraud_label,transaction_type_cat,payment_mode_cat,device_type_cat,device_location_cat,clasificacion
0,2857,1994.55,1596,21,0,1127.61,0,0.661,6,1,1,0,2,2,-22.484161
